# ComfyUI (GGUF) ? Qwen-Image-Edit-2511 (Colab)

This notebook:
- installs ComfyUI + ComfyUI-Manager + ComfyUI-GGUF
- downloads Qwen-Image-Edit-2511 GGUF, Qwen2.5-VL GGUF, VAE, mmproj, and 8-step LoRA
- launches ComfyUI via Cloudflare Tunnel

Quick start:
1. Run all cells from top to bottom.
2. Open the tunnel URL and load your workflow.
3. If VRAM is tight, switch to smaller GGUF quants.


## 1) Installation
Install ComfyUI, ComfyUI-Manager, ComfyUI-GGUF, and swap for better Colab stability.


In [1]:
# @title 1) Install ComfyUI + Manager + ComfyUI-GGUF (with swap)
import os

# Small protection against RAM-related crashes
if not os.path.exists('/swapfile'):
    print('Creating swap (8GB)...')
    !sudo fallocate -l 8G /swapfile
    !sudo chmod 600 /swapfile
    !sudo mkswap /swapfile
    !sudo swapon /swapfile
    print('Swap enabled.')

# Useful packages
!apt-get -y update -qq
!apt-get -y install -qq aria2 ffmpeg

# More predictable CUDA allocator (sometimes helps reduce fragmentation)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

# ComfyUI
if not os.path.exists('/content/ComfyUI'):
    print('Cloning ComfyUI...')
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI

print('Installing python requirements...')
!pip install -U pip
!pip install -r requirements.txt

# Attention acceleration pack (T4-safe)
ENABLE_ACCEL_PACK = True  # Set False to skip optional attention accelerators.

if ENABLE_ACCEL_PACK:
    import sys
    import subprocess

    def pip_try(spec):
        print(f"Installing optional accelerator: {spec}")
        return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec], check=False).returncode == 0

    import torch
    gpu_name = 'CPU'
    cc_major, cc_minor = (0, 0)
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        cc_major, cc_minor = torch.cuda.get_device_capability(0)

    print(f"GPU detected: {gpu_name} (sm{cc_major}{cc_minor})")

    x_ok = pip_try('xformers>=0.0.28.post3')
    print('xformers:', 'OK' if x_ok else 'FAILED (continuing)')

    # FlashAttention-3 is Hopper-only; FlashAttention-2 and SageAttention target Ampere+.
    if cc_major >= 8:
        fa_ok = pip_try('flash-attn>=2.7.0.post2')
        sage_ok = pip_try('sageattention>=2.1.1')
        print('flash-attn:', 'OK' if fa_ok else 'FAILED (continuing)')
        print('sageattention:', 'OK' if sage_ok else 'FAILED (continuing)')
    else:
        print('Skipping flash-attn and sageattention on this GPU: T4/Turing (sm75) uses xformers path.')
else:
    print('Attention acceleration pack disabled by user (ENABLE_ACCEL_PACK=False).')


# ComfyUI-Manager
if not os.path.exists('custom_nodes/ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager

# ComfyUI-GGUF
if not os.path.exists('custom_nodes/ComfyUI-GGUF'):
    print('Installing ComfyUI-GGUF...')
    !git clone https://github.com/city96/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF
    !pip install -r custom_nodes/ComfyUI-GGUF/requirements.txt

# ComfyUI-KJNodes
if not os.path.exists('custom_nodes/comfyui-kjnodes'):
    print('Installing ComfyUI-KJNodes...')
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git custom_nodes/comfyui-kjnodes
if os.path.exists('custom_nodes/comfyui-kjnodes/requirements.txt'):
    !pip install -r custom_nodes/comfyui-kjnodes/requirements.txt



# Hugging Face authentication (interactive prompt)
!pip install -q huggingface_hub
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Enter Hugging Face token: ').strip()
if not hf_token:
    raise RuntimeError('Hugging Face token is required for this notebook run.')

login(token=hf_token, add_to_git_credential=False)
os.environ['HUGGINGFACE_TOKEN'] = hf_token
os.environ['HF_TOKEN'] = hf_token
print('HF auth: OK')

# Civitai authentication (for LoRA downloads)
civitai_token = getpass('Enter Civitai API token (optional, press Enter to skip): ').strip()
if civitai_token:
    os.environ['CIVITAI_API_TOKEN'] = civitai_token
    print('Civitai auth: OK')
else:
    print('Civitai token not set. Civitai downloads may fail (403).')

print('Done.')


Creating swap (8GB)...
Setting up swapspace version 1, size = 8 GiB (8589930496 bytes)
no label, UUID=3290b089-7fba-45e3-b8de-ba7fec7bb64b
swapon: /swapfile: swapon failed: Invalid argument
Swap enabled.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...

## 2) Quant Settings
Choose GGUF quant defaults and review rough file-size/VRAM estimates.


In [6]:
# @title 2) Settings: GGUF quant selection + size/VRAM table
import math

# ---- Auto quant by VRAM ----
AUTO_QUANT_BY_VRAM = True
AUTO_QUANT_VRAM_FRACTION = 0.9
AUTO_VRAM_FALLBACK_GB = 14.0


# ---- Quant selection (editable) ----
QWEN_EDIT_QUANT = 'Q2_K'   # options: Q2_K, Q3_K_S, Q3_K_M, Q3_K_L, Q4_0, Q4_1, Q4_K_S, Q4_K_M, Q5_0, Q5_1, Q5_K_S, Q5_K_M, Q6_K, Q8_0, BF16, F16

# Text encoder (GGUF)
QWEN_VL_QUANT = 'Q4_K_M'  # options: Q4_K_M, Q8_0, F16

# mmproj for Qwen2.5-VL
DOWNLOAD_MMPROJ = True
MMPROJ_QUANT = 'Q8_0'     # options: Q8_0, f16

# LoRA (8 steps)
DOWNLOAD_LORA = True
LORA_FILE = 'Qwen-Image-Edit-Lightning-8steps-V1.0-bf16.safetensors'

# If VRAM is limited, keep this enabled to offload text encoder to CPU/RAM
OFFLOAD_TEXT_ENCODER = False

# ---- Size tables (GB) from Hugging Face listings ----
# Qwen-Image-Edit-2511 GGUF (unsloth)
QWEN_EDIT_SIZES_GB = {
  'Q2_K': 7.47,
  'Q3_K_S': 9.22,
  'Q3_K_M': 9.92,
  'Q3_K_L': 10.6,
  'Q4_0': 11.9,
  'Q4_1': 12.8,
  'Q4_K_S': 12.4,
  'Q4_K_M': 13.2,
  'Q5_0': 14.4,
  'Q5_1': 15.4,
  'Q5_K_S': 14.3,
  'Q5_K_M': 15.0,
  'Q6_K': 16.9,
  'Q8_0': 21.8,
  'BF16': 40.9,
  'F16': 40.9
}

# Qwen2.5-VL-7B-Instruct GGUF (ggml-org)
QWEN_VL_SIZES_GB = {
  'Q4_K_M': 4.68,
  'Q8_0': 8.10,
  'F16': 15.2
}

# mmproj sizes (GB)
MMPROJ_SIZES_GB = {
  'Q8_0': 0.853,
  'f16': 1.35
}

# VAE size (GB)
QWEN_EDIT_VAE_GB = 0.254  # official Comfy-Org qwen_image_vae.safetensors

# LoRA size (GB)
LORA_GB = 0.85  # Qwen-Image-Edit-Lightning-8steps-V1.0-bf16




def detect_total_vram_gb(default=AUTO_VRAM_FALLBACK_GB):
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception as e:
        print(f"VRAM detection failed ({e}); using fallback {default:.2f} GB")
    return default


def pick_best_quant(size_dict, budget_gb, overhead=1.10):
    items = sorted(size_dict.items(), key=lambda kv: kv[1])
    fitting = [(k, v) for k, v in items if (v * overhead) <= budget_gb]
    if fitting:
        return fitting[-1][0]
    return items[0][0]


def est_vram_gb(size_gb, overhead=1.10):
    return size_gb * overhead


def print_table(title, d):
    print('\n' + title)
    print('-' * len(title))
    for k, v in d.items():
        print(f"{k:8s}  file~{v:5.2f} GB   VRAM~{est_vram_gb(v):5.2f} GB")


# ---- Auto apply quant choice based on detected VRAM ----
TOTAL_VRAM_GB = detect_total_vram_gb()
AUTO_BUDGET_GB = TOTAL_VRAM_GB * AUTO_QUANT_VRAM_FRACTION
print(f"\nGPU VRAM detected: ~{TOTAL_VRAM_GB:.2f} GB | Auto budget (75%): ~{AUTO_BUDGET_GB:.2f} GB")

if AUTO_QUANT_BY_VRAM:
    lora_side = est_vram_gb(LORA_GB) if DOWNLOAD_LORA else 0.0

    if OFFLOAD_TEXT_ENCODER:
        budget_edit = max(AUTO_BUDGET_GB - est_vram_gb(QWEN_EDIT_VAE_GB) - lora_side, 0.01)
        QWEN_EDIT_QUANT = pick_best_quant(QWEN_EDIT_SIZES_GB, budget_edit)
    else:
        if DOWNLOAD_MMPROJ:
            best_fit = None
            best_any = None
            for e_k, e_v in QWEN_EDIT_SIZES_GB.items():
                for v_k, v_v in QWEN_VL_SIZES_GB.items():
                    for m_k, m_v in MMPROJ_SIZES_GB.items():
                        total = est_vram_gb(e_v) + est_vram_gb(v_v) + est_vram_gb(m_v) + est_vram_gb(QWEN_EDIT_VAE_GB) + lora_side
                        score = (e_v, v_v, m_v)
                        if (best_any is None) or (total < best_any[0]):
                            best_any = (total, e_k, v_k, m_k)
                        if total <= AUTO_BUDGET_GB:
                            if (best_fit is None) or (score > best_fit[0]) or (score == best_fit[0] and total < best_fit[1]):
                                best_fit = (score, total, e_k, v_k, m_k)
            if best_fit is not None:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT, MMPROJ_QUANT = best_fit[2], best_fit[3], best_fit[4]
            else:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT, MMPROJ_QUANT = best_any[1], best_any[2], best_any[3]
        else:
            best_fit = None
            best_any = None
            for e_k, e_v in QWEN_EDIT_SIZES_GB.items():
                for v_k, v_v in QWEN_VL_SIZES_GB.items():
                    total = est_vram_gb(e_v) + est_vram_gb(v_v) + est_vram_gb(QWEN_EDIT_VAE_GB) + lora_side
                    score = (e_v, v_v)
                    if (best_any is None) or (total < best_any[0]):
                        best_any = (total, e_k, v_k)
                    if total <= AUTO_BUDGET_GB:
                        if (best_fit is None) or (score > best_fit[0]) or (score == best_fit[0] and total < best_fit[1]):
                            best_fit = (score, total, e_k, v_k)
            if best_fit is not None:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT = best_fit[2], best_fit[3]
            else:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT = best_any[1], best_any[2]

    print(f"Auto quant active: QWEN_EDIT_QUANT={QWEN_EDIT_QUANT}, QWEN_VL_QUANT={QWEN_VL_QUANT}, MMPROJ_QUANT={MMPROJ_QUANT}")

print_table('Qwen-Image-Edit-2511 GGUF (DiT) quants', QWEN_EDIT_SIZES_GB)
print_table('Qwen2.5-VL-7B-Instruct GGUF (text encoder) quants', QWEN_VL_SIZES_GB)
print_table('mmproj GGUF', MMPROJ_SIZES_GB)

print('\nSelected:')
print('  QWEN_EDIT_QUANT     =', QWEN_EDIT_QUANT)
print('  QWEN_VL_QUANT       =', QWEN_VL_QUANT)
print('  DOWNLOAD_MMPROJ     =', DOWNLOAD_MMPROJ, 'MMPROJ_QUANT =', MMPROJ_QUANT)
print('  DOWNLOAD_LORA       =', DOWNLOAD_LORA, 'LORA_FILE =', LORA_FILE)
print('  OFFLOAD_TEXT_ENCODER =', OFFLOAD_TEXT_ENCODER)

# Rough total (weights only, excludes activations and current resolution)
total = 0.0
total += est_vram_gb(QWEN_EDIT_SIZES_GB[QWEN_EDIT_QUANT])
total += est_vram_gb(QWEN_EDIT_VAE_GB)
if DOWNLOAD_LORA:
    total += est_vram_gb(LORA_GB)
if not OFFLOAD_TEXT_ENCODER:
    total += est_vram_gb(QWEN_VL_SIZES_GB[QWEN_VL_QUANT])
    if DOWNLOAD_MMPROJ:
        total += est_vram_gb(MMPROJ_SIZES_GB[MMPROJ_QUANT])

print(f"\nRough VRAM for weights if all kept on GPU at once: ~{total:.2f} GB")
print("Note: real peak VRAM depends on resolution/batch/latents; --lowvram/offload is recommended for T4 10GB.")



GPU VRAM detected: ~39.49 GB | Auto budget (75%): ~35.54 GB
Auto quant active: QWEN_EDIT_QUANT=Q8_0, QWEN_VL_QUANT=Q8_0, MMPROJ_QUANT=Q8_0

Qwen-Image-Edit-2511 GGUF (DiT) quants
--------------------------------------
Q2_K      file~ 7.47 GB   VRAM~ 8.22 GB
Q3_K_S    file~ 9.22 GB   VRAM~10.14 GB
Q3_K_M    file~ 9.92 GB   VRAM~10.91 GB
Q3_K_L    file~10.60 GB   VRAM~11.66 GB
Q4_0      file~11.90 GB   VRAM~13.09 GB
Q4_1      file~12.80 GB   VRAM~14.08 GB
Q4_K_S    file~12.40 GB   VRAM~13.64 GB
Q4_K_M    file~13.20 GB   VRAM~14.52 GB
Q5_0      file~14.40 GB   VRAM~15.84 GB
Q5_1      file~15.40 GB   VRAM~16.94 GB
Q5_K_S    file~14.30 GB   VRAM~15.73 GB
Q5_K_M    file~15.00 GB   VRAM~16.50 GB
Q6_K      file~16.90 GB   VRAM~18.59 GB
Q8_0      file~21.80 GB   VRAM~23.98 GB
BF16      file~40.90 GB   VRAM~44.99 GB
F16       file~40.90 GB   VRAM~44.99 GB

Qwen2.5-VL-7B-Instruct GGUF (text encoder) quants
-------------------------------------------------
Q4_K_M    file~ 4.68 GB   VRAM~ 5.15 GB


## 3) Model Download
Download the required model files (and optional components) into ComfyUI folders.


In [3]:
# @title 3) Download models (Qwen-Image-Edit-2511 GGUF + Qwen2.5-VL GGUF + VAE + LoRA)
import os

COMFY = '/content/ComfyUI'

# Directories
UNET_DIR = f'{COMFY}/models/unet'
DIFF_DIR = f'{COMFY}/models/diffusion_models'
TE_DIR   = f'{COMFY}/models/text_encoders'
CLIP_DIR = f'{COMFY}/models/clip'
VAE_DIR  = f'{COMFY}/models/vae'
LORA_DIR = f'{COMFY}/models/loras'

for d in [UNET_DIR, DIFF_DIR, TE_DIR, CLIP_DIR, VAE_DIR, LORA_DIR]:
    os.makedirs(d, exist_ok=True)


def dl(url, outdir, fname):
    outpath = os.path.join(outdir, fname)
    if os.path.exists(outpath):
        print('Already exists:', outpath)
        return
    print('Downloading:', fname)

    hf_token = os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_TOKEN')
    if hf_token and 'huggingface.co' in url:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --header="Authorization: Bearer {hf_token}" "{url}" -d "{outdir}" -o "{fname}"
    else:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{outdir}" -o "{fname}"

# --- Qwen-Image-Edit-2509 Safetensors (Diffusion Model) ---
edit_fname = 'qwen_image_edit_2509_fp8_e4m3fn.safetensors'
edit_url = 'https://huggingface.co/Comfy-Org/Qwen-Image-Edit_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_edit_2509_fp8_e4m3fn.safetensors'
dl(edit_url, DIFF_DIR, edit_fname)

# --- Qwen2.5-VL-7B FP8 (Text Encoder) ---
vl_fname = 'qwen_2.5_vl_7b_fp8_scaled.safetensors'
vl_url = 'https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors'
dl(vl_url, TE_DIR, vl_fname)

# --- Qwen-Image-Edit-2509 Lightning 4-step LoRA ---
lora_fname = 'Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors'
lora_url = 'https://huggingface.co/lightx2v/Qwen-Image-Lightning/resolve/main/Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors'
dl(lora_url, LORA_DIR, lora_fname)

# Keep the VAE as it is usually shared across versions
vae_fname = 'qwen_image_vae.safetensors'
vae_url = 'https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors'
dl(vae_url, VAE_DIR, vae_fname)

# --- VAE ---
vae_fname = 'qwen_image_vae.safetensors'
vae_url = 'https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors'
dl(vae_url, VAE_DIR, vae_fname)

# --- LoRA (8 steps) ---
if DOWNLOAD_LORA:
    lora_fname = LORA_FILE
    lora_url = f"https://huggingface.co/lightx2v/Qwen-Image-Lightning/resolve/main/{lora_fname}"
    dl(lora_url, LORA_DIR, lora_fname)

print('Done downloading models.')
print('Qwen-Edit :', edit_fname)
print('Qwen2.5-VL:', vl_fname)
print('VAE       :', vae_fname)
if DOWNLOAD_MMPROJ:
    print('mmproj    :', mmproj_fname)
if DOWNLOAD_LORA:
    print('LoRA      :', lora_fname)


# ---- Lenovo UltraReal LoRA pack (Civitai) ----
DOWNLOAD_LENOVO_ULTRAREAL_LORAS = True
CIVITAI_API_TOKEN = os.environ.get('CIVITAI_API_TOKEN', '').strip()  # optional, helps if Civitai returns 403.
CURRENT_BASE_MODELS = ['Qwen']

if DOWNLOAD_LENOVO_ULTRAREAL_LORAS:
    import json
    import subprocess
    import urllib.request
    import urllib.parse

    LORA_DIR = f'{COMFY}/models/loras'
    os.makedirs(LORA_DIR, exist_ok=True)

    def add_civitai_token(url, token):
        if not token:
            return url
        parts = urllib.parse.urlsplit(url)
        query = urllib.parse.parse_qsl(parts.query, keep_blank_values=True)
        if not any(k == 'token' for k, _ in query):
            query.append(('token', token))
        return urllib.parse.urlunsplit((parts.scheme, parts.netloc, parts.path, urllib.parse.urlencode(query), parts.fragment))

    def dl_civitai(url, outdir, fname, token=''):
        outpath = os.path.join(outdir, fname)
        if os.path.exists(outpath):
            print('Already exists:', outpath)
            return True

        final_url = add_civitai_token(url, token)
        print('Downloading LoRA:', fname)
        cmd = ['aria2c', '--console-log-level=error', '-c', '-x', '8', '-s', '8', '-k', '1M', final_url, '-d', outdir, '-o', fname]
        rc = subprocess.run(cmd, check=False).returncode
        if rc != 0 and not token:
            print('  Download failed without token. Set CIVITAI_API_TOKEN env var and retry Cell 3.')
        return rc == 0

    try:
        req = urllib.request.Request(
            'https://civitai.com/api/v1/models/1662740',
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            civitai_model = json.loads(resp.read().decode('utf-8'))

        versions = civitai_model.get('modelVersions', [])
        print('\nLenovo UltraReal LoRA versions found:', len(versions))
        if not CURRENT_BASE_MODELS:
            print('Current notebook base-model tags: (none matched explicitly)')
        else:
            print('Current notebook base-model tags:', ', '.join(CURRENT_BASE_MODELS))

        for mv in versions:
            base = mv.get('baseModel', 'Unknown')
            match = base in CURRENT_BASE_MODELS
            mark = 'MATCH' if match else 'OTHER'
            files = [f for f in mv.get('files', []) if f.get('type') == 'Model']
            print(f"- [{mark}] {base} :: {mv.get('name', 'Unnamed version')} ({len(files)} file(s))")

        ok_count = 0
        total_count = 0
        matched_versions = [mv for mv in versions if mv.get('baseModel', 'Unknown') in CURRENT_BASE_MODELS]

        if not matched_versions:
            print('No MATCH versions found for this notebook base-model tag set. Skipping LoRA download.')

        for mv in matched_versions:
            for fobj in mv.get('files', []):
                if fobj.get('type') != 'Model':
                    continue
                total_count += 1
                raw_name = (fobj.get('name') or '').strip()
                base_tag = ''.join(ch if ch.isalnum() else '_' for ch in str(mv.get('baseModel', 'unknown'))).strip('_').lower() or 'unknown'
                fname = f"lenovo_ultrareal_{base_tag}_v{mv.get('id', 'x')}_f{fobj.get('id', 'x')}.safetensors"
                if raw_name and raw_name != fname:
                    print(f"  Civitai file name: {raw_name} -> saved as {fname}")
                if dl_civitai(fobj.get('downloadUrl', ''), LORA_DIR, fname, CIVITAI_API_TOKEN):
                    ok_count += 1

        print(f"\nLenovo UltraReal LoRA download summary (MATCH only): {ok_count}/{total_count} files ready in {LORA_DIR}")

    except Exception as e:
        print('Failed to fetch Lenovo UltraReal LoRA metadata from Civitai:', e)
        print('Tip: this may require CIVITAI_API_TOKEN or temporary Civitai availability.')


Downloading: qwen-image-edit-2511-Q8_0.gguf

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
9b62ed|OK  |   359MiB/s|/content/ComfyUI/models/unet/qwen-image-edit-2511-Q8_0.gguf

Status Legend:
(OK):download completed.
Downloading: Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
21a964|OK  |   374MiB/s|/content/ComfyUI/models/text_encoders/Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf

Status Legend:
(OK):download completed.
Downloading: mmproj-Qwen2.5-VL-7B-Instruct-Q8_0.gguf

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
dd44ae|OK  |   168MiB/s|/content/ComfyUI/models/text_encoders/mmproj-Qwen2.5-VL-7B-Instruct-Q8_0.gguf

Status Legend:
(OK):download completed.
Downloading: qwen_image_vae.safetensors

Download Res

## 4) Download Check
Verify downloaded file sizes and calculate a rough VRAM estimate from real files.


In [4]:
# @title 4) Verify downloads: actual file sizes and VRAM estimate
import os


def size_gb(path):
    return os.path.getsize(path) / (1024**3)


def est_vram_from_file(path, overhead=1.10):
    return size_gb(path) * overhead


paths = []
paths.append(('/content/ComfyUI/models/unet', f"qwen-image-edit-2511-{QWEN_EDIT_QUANT}.gguf"))
paths.append(('/content/ComfyUI/models/text_encoders', f"Qwen2.5-VL-7B-Instruct-{QWEN_VL_QUANT}.gguf"))
paths.append(('/content/ComfyUI/models/vae', 'qwen_image_vae.safetensors'))
if DOWNLOAD_MMPROJ:
    paths.append(('/content/ComfyUI/models/text_encoders', f"mmproj-Qwen2.5-VL-7B-Instruct-{MMPROJ_QUANT}.gguf"))
if DOWNLOAD_LORA:
    paths.append(('/content/ComfyUI/models/loras', LORA_FILE))

print('Files:')
total_vram = 0.0
for d, f in paths:
    p = os.path.join(d, f)
    if not os.path.exists(p):
        print('MISSING:', p)
        continue
    s = size_gb(p)
    v = est_vram_from_file(p)
    total_vram += v
    print(f"- {p}\n  size~{s:.2f} GB  -> VRAM(weights)~{v:.2f} GB")

print(f"\nSum VRAM(weights) if all kept on GPU at once: ~{total_vram:.2f} GB")
print('Reminder: real peak VRAM will be higher (resolution/batch/latents).')


Files:
- /content/ComfyUI/models/unet/qwen-image-edit-2511-Q8_0.gguf
  size~20.27 GB  -> VRAM(weights)~22.29 GB
- /content/ComfyUI/models/text_encoders/Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf
  size~4.36 GB  -> VRAM(weights)~4.80 GB
- /content/ComfyUI/models/vae/qwen_image_vae.safetensors
  size~0.24 GB  -> VRAM(weights)~0.26 GB
- /content/ComfyUI/models/text_encoders/mmproj-Qwen2.5-VL-7B-Instruct-Q8_0.gguf
  size~0.79 GB  -> VRAM(weights)~0.87 GB
- /content/ComfyUI/models/loras/Qwen-Image-Edit-Lightning-8steps-V1.0-bf16.safetensors
  size~0.79 GB  -> VRAM(weights)~0.87 GB

Sum VRAM(weights) if all kept on GPU at once: ~29.10 GB
Reminder: real peak VRAM will be higher (resolution/batch/latents).


## 5) Launch ComfyUI
Start ComfyUI and use the Cloudflare Tunnel URL shown in the output.


In [7]:
# @title 5) Launch ComfyUI (Cloudflare Tunnel) - stable mode
import subprocess, threading, time, socket, os, re, sys, urllib.request

%cd /content/ComfyUI

# Install cloudflared (if not installed yet)
if not os.path.exists("cloudflared-linux-amd64.deb"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb


def wait_port(host="127.0.0.1", port=8188, timeout=240):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    return False


def verify_tunnel_url(url, timeout=10):
    base = url.rstrip('/')
    candidates = [
        base + '/',
        base + '/api/system_stats',
        base + '/object_info',
    ]
    for c in candidates:
        try:
            req = urllib.request.Request(c, headers={'User-Agent': 'Mozilla/5.0'}, method='GET')
            with urllib.request.urlopen(req, timeout=timeout) as r:
                code = r.getcode()
                if 200 <= code < 500:
                    return True
        except Exception:
            pass
    return False


def wait_reachable_stable(url, proc, warmup_timeout=180, stable_successes=3, probe_interval=2.5):
    t0 = time.time()
    ok_streak = 0
    while time.time() - t0 < warmup_timeout:
        if proc.poll() is not None:
            return False
        if verify_tunnel_url(url, timeout=10):
            ok_streak += 1
            if ok_streak >= stable_successes:
                return True
        else:
            ok_streak = 0
        time.sleep(probe_interval)
    return False


def start_cloudflare_tunnel_once(port=8188, protocol='http2', read_timeout=150, warmup_timeout=180):
    print(f"\nStarting Cloudflare Quick Tunnel (protocol={protocol})...\n")
    sys.stdout.flush()

    cmd = [
        'cloudflared', 'tunnel',
        '--no-autoupdate',
        '--url', f'http://127.0.0.1:{port}',
        '--protocol', protocol,
        '--loglevel', 'info',
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tunnel_patterns = [
        re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com', re.I),
        re.compile(r'https://[a-z0-9-]+\.cfargotunnel\.com', re.I),
    ]
    ignore_patterns = [
        re.compile(r'https://www\.cloudflare\.com/website-terms/?', re.I),
    ]

    t0 = time.time()
    url = None

    while time.time() - t0 < read_timeout:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            time.sleep(0.1)
            continue

        s = line.strip()

        if any(ip.search(s) for ip in ignore_patterns):
            continue

        for pat in tunnel_patterns:
            m = pat.search(s)
            if m:
                url = m.group(0)
                break
        if url:
            break

    if not url:
        print('Failed to parse tunnel URL.')
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except Exception:
                proc.kill()
        return None, None

    print(f'Found tunnel URL: {url}')
    print('Waiting for Cloudflare propagation and stable readiness...')

    if wait_reachable_stable(url, proc, warmup_timeout=warmup_timeout, stable_successes=3, probe_interval=2.5):
        print('\n--------------------------------------------------')
        print('YOUR LINK:', url)
        print('--------------------------------------------------\n')
        return proc, url

    print('Tunnel URL did not become stably reachable in time.')
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
    return None, url


def start_tunnel_with_retries(port=8188, max_attempts=6):
    protocols = ['http2', 'quic']
    for attempt in range(1, max_attempts + 1):
        proto = protocols[(attempt - 1) % len(protocols)]
        print(f"\n== Tunnel attempt {attempt}/{max_attempts} ==")
        proc, url = start_cloudflare_tunnel_once(
            port=port,
            protocol=proto,
            read_timeout=150,
            warmup_timeout=180,
        )
        if proc is not None and url:
            return proc, url
        time.sleep(min(8, 2 + attempt))
    return None, None


def stop_proc_safely(proc):
    if proc is None:
        return
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()


def tunnel_thread(port=8188):
    if not wait_port('127.0.0.1', port, timeout=240):
        print('Timed out waiting for ComfyUI port', port)
        return

    print('\nComfyUI port is open. Creating stable tunnel...\n')

    while True:
        proc, url = start_tunnel_with_retries(port=port, max_attempts=6)
        if proc is None:
            print('Failed to establish a reachable Cloudflare tunnel automatically.')
            print('Rerun this cell to retry with a fresh tunnel session.')
            return

        unhealthy_streak = 0
        while proc.poll() is None:
            time.sleep(15)
            if verify_tunnel_url(url, timeout=8):
                unhealthy_streak = 0
            else:
                unhealthy_streak += 1
                print(f'Cloudflare tunnel health check failed ({unhealthy_streak}/3).')
                if unhealthy_streak >= 3:
                    print('Tunnel became unhealthy. Recreating...')
                    stop_proc_safely(proc)
                    break

        if proc.poll() is not None:
            rc = proc.returncode
            print(f'Cloudflare tunnel exited (code={rc}). Recreating...')


threading.Thread(target=tunnel_thread, daemon=True, args=(8188,)).start()

print('Starting ComfyUI... link will appear above.')
!python main.py --dont-print-server --port 8188 --lowvram --preview-method auto


/content/ComfyUI
Starting ComfyUI... link will appear above.
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-03-01 19:44:40.947
** Platform: Linux
** Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log

Prestartup times for custom nodes:
   3.7 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager

Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailab